# ML-07 — Baseline Action Score and Top-10 Review

**Lane 4 — CTR / Engagement Opportunity Scoring** (carried over from Weeks 1–3). Three small things in one notebook: two signal checks, one hand-written rule that writes a ranked queue, and a skeptical read of my own top ten. This is the baseline my Week-5 model has to beat.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load `building-baselines` + `flyrank/flyrank-data`.

In [6]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

CSV = "data/raw/content_refresh_anonymized.csv"
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / CSV).exists()), None)
if root is None:
    root = Path.cwd()
    df = pd.read_csv("https://raw.githubusercontent.com/nothaziq/FlyRank-ML-Week1/main/" + CSV)
else:
    df = pd.read_csv(root / CSV)
OUT = root / "work" / "outputs"
OUT.mkdir(parents=True, exist_ok=True)

BANDS = [0, 3, 5, 7, 10, 15, 20]
BAND_LABELS = ["0-3", "3-5", "5-7", "7-10", "10-15", "15-20"]
IMP_FLOOR, MIN_EXPECTED_CLICKS, SHORTFALL = 500, 10, 0.5
metrics = {}

print(f"{len(df):,} rows x {df.shape[1]} columns, {df.client_id.nunique()} clients (rates such as ctr are x100 percentages)")

30,000 rows x 44 columns, 32 clients (rates such as ctr are x100 percentages)


## 1. Two signal checks, then my rule

Both checks use only the trailing-90-day snapshot columns (`impressions_90d`, `clicks_90d`, `avg_position`). Bands are built from `avg_position`, not from the CSV's `position_tier`, because rows with `avg_position == 0` ("no data") are labelled `top_3` there — checked in the first cell below.

### Signal 1 (flag-linked): CTR falls as position gets worse

FlyRank's `low_ctr_visible_page` flag (impressions ≥ 500, position ≤ 20, `ctr < 0.5`) applies one CTR bar to every page and sends it to `refresh_and_review_ctr`. My rule does the opposite: it compares a page only with pages at a similar position. If CTR did not fall with position, that would be pointless — so check it. Universe: `0 < avg_position ≤ 20` and `impressions_90d ≥ 500`, the same pages the flag looks at.

In [7]:
no_pos = df.avg_position == 0
print(f"avg_position == 0 rows: {no_pos.sum():,}, all labelled position_tier = {df.loc[no_pos, 'position_tier'].unique().tolist()}")

vis = df[(df.avg_position > 0) & (df.avg_position <= 20) & (df.impressions_90d >= IMP_FLOOR)].copy()
vis["band"] = pd.cut(vis.avg_position, BANDS, labels=BAND_LABELS)
t1 = vis.groupby("band", observed=True).agg(
    n=("ctr", "size"), median_ctr=("ctr", "median"),
    clicks=("clicks_90d", "sum"), imps=("impressions_90d", "sum"),
    flat_flag_pct=("ctr", lambda s: (s < 0.5).mean() * 100),
)
t1["pooled_ctr"] = t1.clicks / t1.imps * 100
display(t1[["n", "median_ctr", "pooled_ctr", "flat_flag_pct"]].round(2))

flat_hits = int((vis.ctr < 0.5).sum())
med = t1.median_ctr
falls_beyond_3 = med.iloc[1:].is_monotonic_decreasing
top_is_highest = med.iloc[0] >= med.iloc[1:].max()
verdict1 = "CONFIRMED" if falls_beyond_3 and top_is_highest else "MIXED" if falls_beyond_3 else "FALSE"
print(f"n = {len(vis):,} | flat 0.5 bar fires on {flat_hits:,} pages ({flat_hits / len(vis):.1%})")
print("Signal 1 verdict:", verdict1)
metrics["signal_ctr_vs_position"] = {"verdict": verdict1, "n": len(vis), "flag": "low_ctr_visible_page",
                                     "flat_flag_hits": flat_hits, "flat_flag_share": round(flat_hits / len(vis), 4),
                                     "median_ctr_by_band": med.round(3).to_dict()}

avg_position == 0 rows: 1,205, all labelled position_tier = ['top_3']


,n,median_ctr,pooled_ctr,flat_flag_pct
band,,,,
0-3,480,0.20,0.49,75.00
3-5,1713,0.33,0.46,69.47
5-7,2484,0.24,0.29,81.76
7-10,2887,0.19,0.29,82.72
10-15,2697,0.18,0.36,84.91
15-20,1762,0.16,0.34,85.13


n = 12,023 | flat 0.5 bar fires on 9,759 pages (81.2%)
Signal 1 verdict: MIXED


**Verdict: MIXED.** From positions 3–5 outward the assumption holds cleanly: median CTR steps down from 0.33 to 0.24, 0.19, 0.18 and 0.16 as position worsens. But it does not hold at the very top: pages at positions 0–3 (median 0.20, n = 480) do not beat pages at 3–5 (0.33, n = 1,713), and the click-weighted `pooled_ctr` column says the same — the top is flat, not rising. I have not tested why (average position blends many queries; branded queries; SERP features are all candidates), so I only call it observed.

What this does to the rule: use position **bands** (the CSV's `page_1` tier spans positions 3–10, where median CTR falls from 0.33 to 0.19), take each band's own median instead of assuming a smooth curve, and treat the 0–3 norm as shaky. It also shows the flat flag is useless for prioritising: the one 0.5 bar fires on 81% of these pages, and more often the worse the position from 3–5 outward (69% → 85%) — it mostly re-discovers position.

### Signal 2: CTR needs volume behind it

My rule compares a page's CTR with a band norm. That only means something if the page has enough impressions for the click count not to be chance. Check what share of pages have zero clicks at each volume level, next to what pure chance would give if every page earned its band's typical CTR (`chance_zero_pct`, Poisson). Universe: `0 < avg_position ≤ 20`, all volumes, so the floor itself is visible.

In [8]:
e = df[(df.avg_position > 0) & (df.avg_position <= 20)].copy()
e["band"] = pd.cut(e.avg_position, BANDS, labels=BAND_LABELS)
e["typical"] = e.band.map(med.to_dict()).astype(float)
e["chance_zero"] = np.exp(-e.impressions_90d * e.typical / 100)
VOL_EDGES = [0, 100, 250, 500, 1000, 2000, 5000, 20000, np.inf]
VOL_LABELS = ["<100", "100-250", "250-500", "500-1k", "1k-2k", "2k-5k", "5k-20k", "20k+"]
e["vol"] = pd.cut(e.impressions_90d, VOL_EDGES, labels=VOL_LABELS, right=False)
t2 = e.groupby("vol", observed=True).agg(
    n=("ctr", "size"), median_clicks=("clicks_90d", "median"), median_ctr=("ctr", "median"),
    zero_click_pct=("clicks_90d", lambda s: (s == 0).mean() * 100),
    chance_zero_pct=("chance_zero", lambda s: s.mean() * 100),
)
display(t2.round(2))

zc = t2.zero_click_pct
verdict2 = "CONFIRMED" if zc.is_monotonic_decreasing and zc.loc["500-1k"] >= 25 else "MIXED"
print(f"n = {len(e):,} | zero-click share at the 500-impression floor bucket: {zc.loc['500-1k']:.0f}%")
print("Signal 2 verdict:", verdict2)
metrics["signal_volume_noise"] = {"verdict": verdict2, "n": len(e), "zero_click_pct_by_volume": zc.round(1).to_dict()}

,n,median_clicks,median_ctr,zero_click_pct,chance_zero_pct
vol,,,,,
<100,5165,0.0,0.00,85.85,94.87
100-250,1529,0.0,0.00,65.93,71.79
250-500,1539,0.0,0.00,53.93,49.28
500-1k,2012,1.0,0.15,32.85,25.46
1k-2k,2281,2.0,0.17,16.26,7.17
2k-5k,3034,6.0,0.20,4.98,0.55
5k-20k,3308,23.5,0.26,0.94,0.00
20k+,1388,103.5,0.26,0.07,0.00


n = 20,256 | zero-click share at the 500-impression floor bucket: 33%
Signal 2 verdict: CONFIRMED


**Verdict: CONFIRMED.** The zero-click share falls steadily with volume — from most pages at under 100 impressions to under 1% above 5,000 — and up to about 1,000 impressions it sits within roughly ten points of what chance alone gives, so a low CTR there says little about the page. FlyRank's own 500-impression floor is not enough: a third of pages just above it still have zero clicks. From 1,000 impressions up, zero-click pages start to outnumber chance (16% vs 7% at 1k–2k, 5% vs 0.6% at 2k–5k) — that excess is where real under-capture becomes visible. (`chance_zero_pct` is a rough yardstick built on band medians. Median CTR also climbs with volume, partly for the same reason, which is why the band norms are computed only from pages at or above the 500 floor.)

What this does to the rule: an impressions floor is not enough on its own, so I gate on **expected clicks** — a page is only judged if a typical page in its band would earn at least 10 clicks at its impression level.

### My rule, in plain words

A page on Google's first two pages that gets enough impressions to trust, but earns less than **half** the CTR typical for pages at its position, is a candidate for a title / meta-description review. Rank the candidates by clicks left on the table.

| Piece | Definition | Backed by |
|---|---|---|
| Eligible | `0 < avg_position ≤ 20` and `impressions_90d ≥ 500` | same universe as the FlyRank flag |
| Typical CTR | median CTR of eligible pages in the same position band (`0-3, 3-5, 5-7, 7-10, 10-15, 15-20`) | Signal 1 |
| Trust gate | typical CTR × impressions ≥ 10 expected clicks | Signal 2 |
| Flag | `ctr < 0.5 × typical CTR` | judgment call — "less than half" is easy to read |
| **Score** | expected clicks − actual clicks (clicks below the norm, 90 days); 0 if not flagged | — |
| **Reason code** (the only one) | `ctr_below_position_norm` (unflagged rows: `none`) | — |
| **Action label** | `review_title_meta` (unflagged rows: `monitor`) | — |

The rule recomputes CTR from clicks ÷ impressions rather than using the CSV's 2-decimal `ctr` (band medians differ by less than 0.005 either way); the flat-flag comparison in Signal 1 uses the CSV column because that is what the FlyRank flag reads.

**Inputs the rule may see:** `impressions_90d`, `clicks_90d`, `avg_position` — nothing else. **Kept out on purpose:** `trend_direction`, `trend_pct`, `is_declining_label` (the label chain), the `*_last_30d` / `*_prev_30d` window pairs they are built from, and any product flag or health score. Section 4 proves the score does not depend on them.

## 2. Build the ranked queue (writes the CSV)

In [9]:
RULE_INPUTS = ["impressions_90d", "clicks_90d", "avg_position"]

def score_queue(frame):
    x = frame[RULE_INPUTS].copy()
    x["ctr"] = x.clicks_90d / x.impressions_90d * 100
    x["band"] = pd.cut(x.avg_position, BANDS, labels=BAND_LABELS)
    eligible = x.band.notna() & (x.impressions_90d >= IMP_FLOOR)
    x["eligible"] = eligible
    x["expected_ctr"] = x.ctr.where(eligible).groupby(x.band, observed=True).transform("median")
    x["expected_clicks"] = x.impressions_90d * x.expected_ctr / 100
    x["flagged"] = eligible & (x.expected_clicks >= MIN_EXPECTED_CLICKS) & (x.ctr < SHORTFALL * x.expected_ctr)
    x["score"] = np.where(x.flagged, x.expected_clicks - x.clicks_90d, 0.0)
    return x

q = score_queue(df)
q.insert(0, "content_id", df.content_id)
q.insert(1, "client_id", df.client_id)
q["reason_code"] = np.where(q.flagged, "ctr_below_position_norm", "none")
q["action_label"] = np.where(q.flagged, "review_title_meta", "monitor")
q = q.sort_values(["score", "impressions_90d"], ascending=False).reset_index(drop=True)
q["rank"] = q.index + 1

QUEUE_COLS = ["rank", "content_id", "client_id", "score", "reason_code", "action_label",
              "impressions_90d", "clicks_90d", "ctr", "avg_position", "band", "expected_ctr", "expected_clicks"]
q[QUEUE_COLS].to_csv(OUT / "baseline_action_score.csv", index=False)

n_elig, n_flag = int(q.eligible.sum()), int(q.flagged.sum())
print(f"Wrote {OUT / 'baseline_action_score.csv'}: {len(q):,} rows ranked")
print(f"eligible {n_elig:,} | flagged {n_flag:,} ({n_flag / n_elig:.1%} of eligible) vs flat FlyRank flag {flat_hits:,} ({flat_hits / n_elig:.1%})")
print(f"top score {q.score.max():,.0f} clicks below norm | flagged rows carry {q.loc[q.flagged, 'impressions_90d'].sum():,} impressions")
q.loc[q.flagged, [c for c in QUEUE_COLS if c not in ("content_id", "client_id")]].head(5).round(2)

Wrote C:\Users\muham\OneDrive\Desktop\Flyrank\FlyRank-ML-Week1\work\outputs\baseline_action_score.csv: 30,000 rows ranked
eligible 12,023 | flagged 881 (7.3% of eligible) vs flat FlyRank flag 9,759 (81.2%)
top score 943 clicks below norm | flagged rows carry 18,912,116 impressions


,rank,score,reason_code,action_label,impressions_90d,clicks_90d,ctr,avg_position,band,expected_ctr,expected_clicks
0,1,942.77,ctr_below_position_norm,review_title_meta,517715,741,0.14,4.2,3-5,0.33,1683.77
1,2,484.87,ctr_below_position_norm,review_title_meta,213963,211,0.10,4.7,3-5,0.33,695.87
2,3,480.80,ctr_below_position_norm,review_title_meta,272144,75,0.03,2.3,0-3,0.20,555.80
3,4,413.47,ctr_below_position_norm,review_title_meta,295097,154,0.05,7.3,7-10,0.19,567.47
4,5,401.29,ctr_below_position_norm,review_title_meta,208678,0,0.00,9.7,7-10,0.19,401.29


In [10]:
el = q[q.eligible].assign(vs_norm=lambda d: d.ctr / d.expected_ctr)
clients = el.groupby("client_id").agg(eligible=("vs_norm", "size"), median_ctr_vs_norm=("vs_norm", "median"), flagged=("flagged", "sum"))
clients["share_of_eligible"] = clients.eligible / clients.eligible.sum()
clients["flag_rate"] = clients.flagged / clients.eligible
BIG = clients.eligible.idxmax()
TYPICAL = q.groupby("band", observed=True).expected_ctr.first()

## 3. Top-10 review

One line per row: the action, why it is there, and what would make it wrong. The "wrong if" text comes from checks computed on each row, in this order: a measurement check (zero clicks on huge volume, or GA4 sessions far below GSC clicks), how fragile the flag is to the norm I chose (does it survive the neighbouring bands' lower norm?), the shaky 0–3 norm from Signal 1, and the largest client's weight (Section 4). If none fires, the fallback is the risk that always exists: the results page itself absorbing the click, phrased by the page's `main_intent`. I also tried a target-keyword-volume check and dropped it — Section 4 shows why.

In [11]:
top = q.head(10).merge(df[["content_id", "sessions_90d", "main_intent"]], on="content_id", how="left")
big_note = f"the largest client ({clients.share_of_eligible[BIG]:.0%} of eligible pages, median CTR {clients.median_ctr_vs_norm[BIG]:.2f}x the norm)"

def risks(r):
    out = []
    i = BAND_LABELS.index(r.band)
    caution = min(TYPICAL[b] for b in BAND_LABELS[max(i - 1, 0): i + 2])
    margin = r.ctr / (SHORTFALL * caution)
    if r.clicks_90d == 0 and r.impressions_90d >= 100_000:
        out.append(("measurement", f"{r.impressions_90d:,} impressions, 0 clicks and only {r.sessions_90d} GA4 sessions look more like unseen or non-human impressions than a weak title, so verify the data first"))
    elif r.clicks_90d > 0 and r.sessions_90d <= 0.5 * r.clicks_90d:
        out.append(("ga4_gap", f"GA4 recorded {r.sessions_90d} sessions for {r.clicks_90d} GSC clicks, so the two sources disagree and the clicks need checking before anyone acts"))
    if margin >= 1:
        out.append(("norm_fragile", f"it is flagged only because band {r.band} has a high norm ({TYPICAL[r.band]:.2f}); under a neighbouring band's lower norm ({caution:.2f}) it would not be flagged"))
    elif margin >= 0.8:
        out.append(("norm_fragile", f"it only just clears the flag: with the lowest neighbouring norm ({caution:.2f}) it sits {1 - margin:.0%} inside the threshold"))
    if r.band == "0-3":
        out.append(("shaky_top_band", "the 0-3 norm is itself unreliable (pages at 3-5 out-earn it), so 'typical' may be off in either direction"))
    if r.client_id == BIG:
        out.append(("big_client", f"it belongs to {big_note}, so the cause may be site-wide, not this page"))
    if not out:
        serp = ("the query is answered on the results page (snippet or AI summary), so no title change wins the click" if r.main_intent == "informational"
                else "ads and shopping units crowd this kind of query, so a better title would not move CTR")
        out.append(("serp", serp))
    return out[:2]

top["risks"] = [risks(r) for r in top.itertuples()]
lines = []
for r in top.itertuples():
    why = (f"{r.impressions_90d:,} impressions at position {r.avg_position:.1f} but {r.clicks_90d:,} clicks "
           f"(CTR {r.ctr:.2f}% vs {r.expected_ctr:.2f}% typical for band {r.band}), about {r.score:,.0f} clicks below the norm in 90 days")
    lines.append(f"**#{r.rank} · `{r.action_label}`** — {why}. *Wrong if:* " + "; ".join(t for _, t in r.risks) + ".")
display(Markdown("\n\n".join(lines)))

**#1 · `review_title_meta`** — 517,715 impressions at position 4.2 but 741 clicks (CTR 0.14% vs 0.33% typical for band 3-5), about 943 clicks below the norm in 90 days. *Wrong if:* it is flagged only because band 3-5 has a high norm (0.33); under a neighbouring band's lower norm (0.20) it would not be flagged.

**#2 · `review_title_meta`** — 213,963 impressions at position 4.7 but 211 clicks (CTR 0.10% vs 0.33% typical for band 3-5), about 485 clicks below the norm in 90 days. *Wrong if:* it only just clears the flag: with the lowest neighbouring norm (0.20) it sits 3% inside the threshold.

**#3 · `review_title_meta`** — 272,144 impressions at position 2.3 but 75 clicks (CTR 0.03% vs 0.20% typical for band 0-3), about 481 clicks below the norm in 90 days. *Wrong if:* the 0-3 norm is itself unreliable (pages at 3-5 out-earn it), so 'typical' may be off in either direction.

**#4 · `review_title_meta`** — 295,097 impressions at position 7.3 but 154 clicks (CTR 0.05% vs 0.19% typical for band 7-10), about 413 clicks below the norm in 90 days. *Wrong if:* it belongs to the largest client (37% of eligible pages, median CTR 0.80x the norm), so the cause may be site-wide, not this page.

**#5 · `review_title_meta`** — 208,678 impressions at position 9.7 but 0 clicks (CTR 0.00% vs 0.19% typical for band 7-10), about 401 clicks below the norm in 90 days. *Wrong if:* 208,678 impressions, 0 clicks and only 6 GA4 sessions look more like unseen or non-human impressions than a weak title, so verify the data first; it belongs to the largest client (37% of eligible pages, median CTR 0.80x the norm), so the cause may be site-wide, not this page.

**#6 · `review_title_meta`** — 223,271 impressions at position 7.8 but 70 clicks (CTR 0.03% vs 0.19% typical for band 7-10), about 359 clicks below the norm in 90 days. *Wrong if:* the query is answered on the results page (snippet or AI summary), so no title change wins the click.

**#7 · `review_title_meta`** — 201,111 impressions at position 5.7 but 219 clicks (CTR 0.11% vs 0.24% typical for band 5-7), about 262 clicks below the norm in 90 days. *Wrong if:* it is flagged only because band 5-7 has a high norm (0.24); under a neighbouring band's lower norm (0.19) it would not be flagged; it belongs to the largest client (37% of eligible pages, median CTR 0.80x the norm), so the cause may be site-wide, not this page.

**#8 · `review_title_meta`** — 119,217 impressions at position 7.0 but 26 clicks (CTR 0.02% vs 0.24% typical for band 5-7), about 259 clicks below the norm in 90 days. *Wrong if:* the query is answered on the results page (snippet or AI summary), so no title change wins the click.

**#9 · `review_title_meta`** — 147,670 impressions at position 6.4 but 97 clicks (CTR 0.07% vs 0.24% typical for band 5-7), about 257 clicks below the norm in 90 days. *Wrong if:* it belongs to the largest client (37% of eligible pages, median CTR 0.80x the norm), so the cause may be site-wide, not this page.

**#10 · `review_title_meta`** — 140,079 impressions at position 7.6 but 16 clicks (CTR 0.01% vs 0.19% typical for band 7-10), about 253 clicks below the norm in 90 days. *Wrong if:* GA4 recorded 8 sessions for 16 GSC clicks, so the two sources disagree and the clicks need checking before anyone acts.

## 4. Weak picks + leakage check

### Concentration: is this a page queue or a client queue?

In [12]:
show = clients.sort_values("eligible", ascending=False).head(5).copy()
show.index = [f"client {c}" for c in "ABCDE"]
display(show[["eligible", "share_of_eligible", "median_ctr_vs_norm", "flagged", "flag_rate"]].round(3))

flagged_q = q[q.flagged]
big_share_flagged = (flagged_q.client_id == BIG).mean()
big_share_top100 = (flagged_q.head(100).client_id == BIG).mean()
print(f"largest client: {clients.share_of_eligible[BIG]:.0%} of eligible pages, {big_share_flagged:.0%} of flagged pages, {big_share_top100:.0%} of the top 100, {(top.client_id == BIG).sum()} of the top 10")
print(f"clients with at least one flagged page: {flagged_q.client_id.nunique()} of {q.client_id.nunique()}")

,eligible,share_of_eligible,median_ctr_vs_norm,flagged,flag_rate
client A,4395,0.366,0.796,524,0.119
client B,1544,0.128,1.535,57,0.037
client C,1511,0.126,1.044,95,0.063
client D,1060,0.088,1.021,41,0.039
client E,594,0.049,1.314,29,0.049


largest client: 37% of eligible pages, 59% of flagged pages, 60% of the top 100, 4 of the top 10
clients with at least one flagged page: 19 of 32


In [13]:
print("risk tags on the top 10 (a row can carry up to two):")
display(pd.DataFrame({"rank": top["rank"], "tags": [", ".join(t for t, _ in rk) for rk in top.risks]}).set_index("rank").T)

gate = q[q.eligible & (q.expected_clicks >= MIN_EXPECTED_CLICKS)].merge(df[["content_id", "search_volume"]], on="content_id")
gate["dwarfs_keyword"] = gate.impressions_90d > 50 * gate.search_volume.fillna(np.inf)
mismatch = gate.groupby("flagged").dwarfs_keyword.agg(n="size", share="mean")
display(mismatch.round(3))
metrics["keyword_volume_check"] = {"flagged_share": round(float(mismatch.share[True]), 3), "unflagged_share": round(float(mismatch.share[False]), 3)}

risk tags on the top 10 (a row can carry up to two):


rank,1,2,3,4,5,6,7,8,9,10
tags,norm_fragile,norm_fragile,shaky_top_band,big_client,"measurement, big_client",serp,"norm_fragile, big_client",serp,big_client,ga4_gap


,n,share
flagged,,
False,3934,0.950
True,881,0.885


### Weak picks I would challenge

The top-ten review found weak picks, so the queue is not something to hand over unread:

- **#5 is the weakest.** 208,678 impressions, zero clicks and six GA4 sessions. Among the 881 flagged pages, 29 have zero clicks and this is the only one with 100k+ impressions. It looks like a measurement problem (unseen or non-human impressions) rather than a title problem, and it ranks fifth only because the score is "clicks below the norm", which rewards raw volume. **#10** is a data question too: GA4 recorded half as many sessions as GSC clicks.
- **#1 rests on my norm.** It is flagged only because band 3–5 holds the highest norm in the data (0.33) — the local peak that made Signal 1 MIXED. Its CTR (0.14%) is still below both neighbouring bands' typical values, but not by half, so I read it as a real shortfall with a fragile flag. **#7** has the same sensitivity, and **#2** clears the threshold by 3%.
- **It is partly a client queue, not only a page queue.** One client holds 37% of eligible pages but 59% of flagged pages and 60% of the top 100, with 4 of the top 10. Its pages sit at 0.80× the pooled norm and its flag rate is 11.9% against 3.7–6.3% for the next four clients. The pooled norm partly *is* this client, so a reviewer should ask whether the cause is site-wide before rewriting page by page.
- **Tested and dropped: target-keyword volume.** I expected "impressions dwarf the target keyword's volume" to explain flagged pages. It does not: it is true of 88.5% of flagged and 95.0% of unflagged pages, so it separates nothing (and `search_volume` is not on the same scale as 90-day impressions).

The most defensible picks are **#6 and #8**: they survive the neighbouring bands' norms, carry no measurement warning and are not the largest client's. I did not tune the 10-click gate or the 0.5 ratio against this review, and I am leaving the rule frozen so Week 5 has a fixed target. Fixes I would try later, not now: a sanity check on zero-click flags and a client-relative norm.

### Leakage check

The rule function receives only `impressions_90d`, `clicks_90d` and `avg_position`. Scrambling every label and window column that exists in the CSV leaves every score unchanged, and the 1,205 pages with no position data (`avg_position == 0`) are never flagged.

In [14]:
BANNED = ["trend_direction", "trend_pct", "is_declining_label"] + [c for c in df.columns if c.endswith(("_last_30d", "_prev_30d"))]
overlap = sorted(set(RULE_INPUTS) & set(BANNED))
print("rule inputs:", RULE_INPUTS)
print("banned columns used by the rule:", overlap or "none")

present = [c for c in BANNED if c in df.columns]
scrambled = df.copy()
for c in present:
    scrambled[c] = scrambled[c].sample(frac=1, random_state=0).to_numpy()
unchanged = score_queue(scrambled).score.equals(score_queue(df).score)
print(f"scrambled {len(present)} label/window columns present in the CSV (seed 0; is_declining_label only exists after the prep script), scores unchanged: {unchanged}")

pos0 = q.loc[q.avg_position == 0]
print(f"avg_position == 0 rows in queue: {len(pos0):,}, any flagged: {bool(pos0.flagged.any())}")
print("product flags / health scores in this CSV:", [c for c in df.columns if "health" in c or "quick_win" in c] or "none")

rule inputs: ['impressions_90d', 'clicks_90d', 'avg_position']
banned columns used by the rule: none
scrambled 8 label/window columns present in the CSV (seed 0; is_declining_label only exists after the prep script), scores unchanged: True
avg_position == 0 rows in queue: 1,205, any flagged: False
product flags / health scores in this CSV: none


### Lane lock

**Confirming Lane 4 (CTR / Engagement Opportunity Scoring).** Signal 1 shows the position adjustment is not cosmetic — the flat 0.5 bar fires on 81% of eligible pages, while the position-aware rule with a volume gate flags 881 (7%) and gives them an order. Signal 2 gives the rule a principled volume gate. Two things I take into Week 5 rather than hide: there is still no outcome label, so this queue is decision-support ranking, not a validated hit rate; and one client dominates the population, so the model has to be tested on held-out clients with a client-relative norm in the comparison.

Wording note: everything above is observed, measured, directional, decision-support. The queue says where CTR is far below what similar-position pages earn — not that a title rewrite will lift clicks.

## 5. Self-check

In [15]:
queue = pd.read_csv(OUT / "baseline_action_score.csv")
checks = {
    "two signal verdicts recorded, both with n": all(k in metrics for k in ["signal_ctr_vs_position", "signal_volume_noise"]),
    "at least one signal is flag-linked (low_ctr_visible_page)": metrics["signal_ctr_vs_position"]["flag"] == "low_ctr_visible_page",
    "queue has score + ONE reason code + action label": set(queue.reason_code) <= {"ctr_below_position_norm", "none"} and set(queue.action_label) <= {"review_title_meta", "monitor"},
    "queue is ranked (rank 1..N, score non-increasing)": queue["rank"].tolist() == list(range(1, len(queue) + 1)) and queue.score.is_monotonic_decreasing,
    "ten reviewed rows": len(lines) == 10,
    "no label-derived / future-window inputs": not overlap and unchanged,
}
for name, ok in checks.items():
    print("PASS" if ok else "FAIL", "-", name)
assert all(checks.values())

metrics.update({
    "notebook": "w04_baseline_score.ipynb",
    "lane": "Lane 4 - CTR / Engagement Opportunity Scoring",
    "rule": {"inputs": RULE_INPUTS, "bands": BAND_LABELS, "impressions_floor": IMP_FLOOR,
             "min_expected_clicks": MIN_EXPECTED_CLICKS, "shortfall_ratio": SHORTFALL,
             "score": "expected clicks - actual clicks (90d)", "reason_code": "ctr_below_position_norm",
             "action_label": "review_title_meta"},
    "queue": {"rows": len(queue), "eligible": n_elig, "flagged": n_flag,
              "flat_flag_hits_same_universe": flat_hits, "top_score": round(float(queue.score.max()), 1)},
    "concentration": {"largest_client_share_of_eligible": round(float(clients.share_of_eligible[BIG]), 3),
                      "largest_client_share_of_flagged": round(float(big_share_flagged), 3),
                      "largest_client_share_of_top100": round(float(big_share_top100), 3)},
    "leakage_check": {"banned_columns_used": overlap, "scrambled_columns_scores_unchanged": bool(unchanged), "seed": 0},
})
with open(OUT / "w04_baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))
print("wrote", OUT / "w04_baseline_metrics.json")

PASS - two signal verdicts recorded, both with n
PASS - at least one signal is flag-linked (low_ctr_visible_page)
PASS - queue has score + ONE reason code + action label
PASS - queue is ranked (rank 1..N, score non-increasing)
PASS - ten reviewed rows
PASS - no label-derived / future-window inputs
wrote C:\Users\muham\OneDrive\Desktop\Flyrank\FlyRank-ML-Week1\work\outputs\w04_baseline_metrics.json


Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.